# AI CUP 2026 圍棋 — 兩個競賽的完整訓練（Kaggle Notebooks）

## 使用前

1. 右側 **Settings → Accelerator → GPU P100**（或 T4 x2）
2. 右側 **Settings → Internet → On**（需手機驗證）
3. **Save Version → Save & Run All (Commit)** 才會背景跑滿 12 小時

## 這份 notebook 會做什麼

| 階段 | 內容 | 估計時間 |
|---|---|---|
| 1–3 | 環境、程式碼、資料、測試 | 10 分 |
| 4 | 手工特徵庫（168 維 x 100 萬局） | 25 分 |
| 5 | **競賽二**：GBDT 集成（LightGBM + XGBoost + CatBoost + HistGB）+ stacking | 60–90 分 |
| 6 | **競賽一**：深度 embedding（Sub-center ArcFace + Set Transformer） | 3–5 小時 |
| 7 | **競賽二**：深度模型（CORN + 棋形品質輔助 head） | 3–5 小時 |
| 8 | 融合 deep + handcrafted，產出最終分數 | 20 分 |

**兩個競賽都訓練。** 目前的離線基準：

| 子題 | 官方 baseline | 純手工特徵 | 目標 |
|---|---:|---:|---|
| 競賽一 玩家辨識 | 0.2845 | 0.6726 | 深度 + 融合後應更高 |
| 競賽二 棋力預測 | 0.4631 | 0.3973 | **先超過 0.4631** |

## 1. 環境

In [ ]:
import torch, os, multiprocessing
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('CPU 核心', multiprocessing.cpu_count())
for m in ('lightgbm','xgboost','catboost'):
    try:
        print(m, __import__(m).__version__)
    except ImportError:
        print(m, '未安裝')

## 2. 程式碼與資料

訓練路徑只用 numpy / pandas / sklearn / torch，Kaggle 全部預裝，不需 pip install。

In [ ]:
%cd /kaggle/working
!test -d AI-CUP-2026-GO || git clone -q https://github.com/ShiYu0318/AI-CUP-2026-GO.git
%cd /kaggle/working/AI-CUP-2026-GO
!git pull -q 2>/dev/null || true

URL = 'https://github.com/AILAB-NDHU/aicup2026-go-dataset/releases/download/v1.0/aicup2026_go_training.tar.xz'
SHA = '959ebc602b9638d665cb063a6269e0d5aef73956b77913e725763eb335ced94b'
!test -d training || wget -q --show-progress $URL
!test -d training || (echo "$SHA  aicup2026_go_training.tar.xz" | sha256sum -c && tar xf aicup2026_go_training.tar.xz)
!du -sh training

## 3. 健全性檢查

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
!python -m pytest -q 2>&1 | tail -3

## 4. 手工特徵庫（約 25 分鐘 / 4 核）

競賽二的 GBDT、深度模型的棋形品質輔助 head、以及最後的融合都需要它。

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
import os, sys, subprocess
if os.path.exists('cache/features.npy'):
    print('特徵庫已存在，略過')
else:
    subprocess.run([sys.executable, 'scripts/build_feature_store.py', '--nproc', '4'], check=True)

## 5. 競賽二：GBDT 集成

LightGBM / XGBoost / CatBoost / HistGB 各自做 **GroupKFold（以玩家分組）** 產生 OOF，
再做加權融合與 stacking。權重以**官方評分函數**搜尋，不是 accuracy。

⚠️ 分組必須是玩家：隨機 KFold 會讓同一位玩家的棋局橫跨 fold，OOF 被身分洩漏污染。

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
!python scripts/train_gbdt_ensemble.py --folds 5

## 6. 競賽一：深度 embedding 訓練

架構：GoTrunk（含 KataGo global pooling bias）→ GeM → BN-Neck → embedding
損失：Sub-center ArcFace（6,405 身分, K=3, dynamic margin）+ batch-hard triplet + 棋力輔助
取樣：P 位玩家 × K 局

**checkpoint 依「未見身分的檢索分數」選**，不是 loss。
先跑探路確認會學，再開正式設定。

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
# 探路：約 30 分鐘
!python scripts/train_player.py --device cuda \
    --width 64 --blocks 6 --embed-dim 256 --snapshots 2 \
    --p 24 --k 4 --steps 3000 \
    --eval-every 500 --eval-candidates 200 --eval-gallery 40 \
    --save-every 500 --workers 4 --max-hours 0.8 \
    --out /kaggle/working/ckpt_player_probe

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
# 正式：加 --resume 可跨 session 接續
!python scripts/train_player.py --device cuda \
    --width 128 --blocks 10 --embed-dim 512 --snapshots 4 \
    --p 32 --k 4 --steps 20000 \
    --eval-every 2000 --eval-candidates 400 --eval-gallery 100 \
    --save-every 500 --workers 4 --max-hours 5.0 \
    --pool set_transformer \
    --out /kaggle/working/ckpt_player

## 7. 競賽二：深度模型訓練

損失對齊評分函數：指數 soft label（相鄰級權重恰為 exp(-1)）+ CORN ordinal head
＋ 棋形品質輔助 head（愚形率等 6 個連續量，提供比 10 類更密集的監督訊號）。

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
!python scripts/train.py --device cuda \
    --width 128 --blocks 10 --embed-dim 512 --snapshots 4 \
    --p 32 --k 8 --steps 20000 \
    --eval-every 1000 --eval-questions 800 \
    --save-every 500 --workers 4 \
    --balance-losses --quality-weight 0.3 \
    --pool set_transformer --max-hours 5.0 \
    --out /kaggle/working/ckpt_rank

## 8. 融合 deep + handcrafted

兩條路徑的誤差結構完全不同：手工特徵看統計習慣，深度模型看盤面。
融合權重在 player-disjoint 的驗證題目上以官方分數搜尋。

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
!python scripts/blend.py --device cuda \
    --player-ckpt /kaggle/working/ckpt_player/best.pt \
    --rank-ckpt   /kaggle/working/ckpt_rank/best.pt \
    --snapshots 4 --seeds 3

## 9. 訓練曲線

In [ ]:
import json, os
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for k, (path, floor, base, title) in enumerate({
        'player': ('/kaggle/working/ckpt_player/history.json', 0.6726, 0.2845, '競賽一 檢索分數'),
        'rank':   ('/kaggle/working/ckpt_rank/history.json',   0.3973, 0.4631, '競賽二 題目分數')}.items()):
    if not os.path.exists(path):
        ax[k].set_title(title + '（尚未訓練）'); continue
    h = json.load(open(path))
    key = 'official_score' if 'player' in path else 'score_expected'
    ax[k].plot([r['step'] for r in h], [r[key] for r in h], marker='o', ms=3)
    ax[k].axhline(floor, ls='--', c='gray', label=f'手工特徵 {floor}')
    ax[k].axhline(base, ls='--', c='crimson', label=f'官方 baseline {base}')
    ax[k].set_title(title); ax[k].set_xlabel('step'); ax[k].legend(fontsize=8); ax[k].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 跨 session 續訓

Kaggle 單次上限 12 小時。做法：

1. 右側 **Add Input → Notebook Output**，掛載這個 notebook 的上一個版本
2. 把 checkpoint 從 `/kaggle/input/<你的-notebook-slug>/ckpt_player` 複製到 `/kaggle/working/ckpt_player`
3. 訓練指令加 `--resume`

```python
import shutil, os
PREV = '/kaggle/input/<你的-notebook-slug>'
for name in ('ckpt_player', 'ckpt_rank'):
    src, dst = f'{PREV}/{name}', f'/kaggle/working/{name}'
    if os.path.isdir(src):
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(src):
            shutil.copy(f'{src}/{f}', dst)
        print('已還原', name)
```

## 判斷成功的標準

**不要看 loss，看每次 eval 印出的官方分數。**

| 子題 | 必須先贏過（手工特徵） | 官方 baseline |
|---|---:|---:|
| 競賽一 | 0.6726 | 0.2845 |
| 競賽二 | 0.3973 | 0.4631 |

若深度模型贏不過手工特徵，是訓練量或實作問題，不是資料難。